# VascuMap Batch Pipeline

**The notebook workflow has been retired.** Run the pipeline from the terminal instead.

## From the repository root

```powershell
cd C:\Users\taylorhearn\git_repos\vascumap
python -m vascumap
```

## From anywhere

```powershell
python C:\Users\taylorhearn\git_repos\vascumap\launch.py
```

> ⚠️ Do **not** run `python -m vascumap` from inside the `vascumap\vascumap\` package folder — Python will pick up `vascumap.py` instead of the package and silently exit. Use one of the commands above.

A launcher window will open. Configure folders, organoid masking, device width, and model checkpoints, then click either **Use GUI (Curation)** or **Run Automatically**. Progress is printed to the terminal.

# VascuMap Batch Pipeline

**The notebook workflow has been retired.**

Run the pipeline from the repository root with:

```powershell
python -m vascumap
```

A launcher window will open. Configure folders, organoid masking, device width, and model checkpoints, then click either **Use GUI (Curation)** or **Run Automatically**. Progress is printed to the terminal.

# VascuMap Batch Pipeline

Two workflow modes:

1. **GUI Curation** — Launch `CurationApp` to review & curate device ROI + organoid mask
   for every image in a napari viewer, then run the full pipeline unattended.
2. **Headless Automatic** — Skip the GUI and run fully automatic device segmentation +
   pipeline on every image.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(r"C:\Users\taylorhearn\git_repos\vascumap\bel_vascumap")))

from liffile import LifFile

from vascumap import VascuMap
from gui_region_detection import CurationApp
from models import Pix2Pix, load_segmentation_model

W0420 20:02:29.978000 602428 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def discover_jobs(source_dir, force_mask=None, require_merged=True):
    """Find .lif/.tif/.tiff image files in source_dir and return (source, files, jobs)."""
    source = Path(source_dir)
    if not source.is_dir():
        raise FileNotFoundError(f"source_dir does not exist: {source}")
    image_files = sorted(p for p in source.iterdir() if p.is_file() and p.suffix.lower() in (".lif", ".tif", ".tiff"))

    def determine_mask_mode(file_path, image_name=""):
        """Infer dark/light/False mask mode from file and image name keywords."""
        if force_mask is not None:
            return force_mask
        combined = (file_path.name + " " + image_name).lower()
        if "marina" in combined and "bead" in combined:
            return "light"
        return "dark" if "marina" in combined else False

    jobs = []
    for file_path in image_files:
        if file_path.suffix.lower() == ".lif":
            try:
                with LifFile(file_path) as lif:
                    for idx, image in enumerate(lif.images):
                        image_name = getattr(image, "name", "")
                        if require_merged and "merged" not in image_name.lower():
                            continue
                        jobs.append((file_path, idx, determine_mask_mode(file_path, image_name), image_name))
            except Exception as exc:
                print(f"[SKIP] {file_path.name}: {exc}")
        else:
            if require_merged and "merged" not in file_path.name.lower():
                continue
            jobs.append((file_path, 0, determine_mask_mode(file_path), file_path.stem))
    return source, image_files, jobs


def expected_output_name(file_path, idx, image_name):
    """Build the output folder name that the pipeline will use for a job."""
    file_path = Path(file_path)
    if file_path.suffix.lower() == ".lif":
        safe_name = image_name.replace("/", "_").replace("\\", "_")
        return f"{file_path.stem}_{safe_name}_img{idx}"
    return file_path.stem


def filter_jobs(jobs, skip_names):
    """Remove jobs whose expected output name is already in skip_names."""
    kept, skipped = [], 0
    for job in jobs:
        file_path, idx, mask_flag, image_name = job
        if expected_output_name(file_path, idx, image_name) in skip_names:
            skipped += 1
        else:
            kept.append(job)
    if skipped:
        print(f"Filtered out {skipped} already-processed images, {len(kept)} remaining.")
    return kept


def run_batch_from_curation(curated_jobs, output_base, save_all_interim=False,
                            model_p2p=None, model_unet=None):
    """Run the full pipeline on curated jobs whose status is 'curated'."""
    results = []
    Path(output_base).mkdir(parents=True, exist_ok=True)
    processable = [j for j in curated_jobs if j.status == "curated"
                   and hasattr(j, 'finalised_outputs') and j.finalised_outputs is not None]
    for i, job in enumerate(processable, 1):
        name = expected_output_name(job.source_path, job.image_index, job.image_name)
        print(f"\n{'='*70}\n[{i}/{len(processable)}] {name}\n{'='*70}")
        try:
            vm = VascuMap(curated_outputs=job.finalised_outputs, model_p2p=model_p2p, model_unet=model_unet)
            vm.image_name = name
            output_dir = Path(output_base) / name
            print(f"  Output → {output_dir}")
            vm.pipeline(output_dir=output_dir, save_all_interim=save_all_interim)
            results.append((name, "OK", ""))
        except Exception as exc:
            print(f"  [SKIP] {exc}")
            results.append((name, "FAILED", str(exc)))
    succeeded = sum(1 for _, s, _ in results if s == "OK")
    skipped_curation = sum(1 for j in curated_jobs if j.status == "skip")
    print(f"\n{'='*70}\nBatch complete: {succeeded}/{len(results)} succeeded, "
          f"{skipped_curation} skipped (curation)")
    for name, status, msg in results:
        if status == "FAILED":
            print(f"  FAILED: {name}: {msg}")
    return results


def run_batch(jobs, output_base, device_width_um, brightfield_channel=0,
              save_all_interim=False, model_p2p=None, model_unet=None,
              start_index=1):
    """Run all jobs in headless automatic mode (no GUI curation)."""
    results = []
    Path(output_base).mkdir(parents=True, exist_ok=True)
    failure_diag_dir = Path(output_base) / "FAILED_diagnostics"
    for i, (file_path, idx, mask_flag, image_name) in enumerate(jobs, start_index):
        name = expected_output_name(file_path, idx, image_name)
        lif_tag = f" (LIF #{idx}: {image_name})" if file_path.suffix.lower() == ".lif" else ""
        print(f"\n{'='*70}\n[{i}/{len(jobs)}] {file_path.name}{lif_tag}  mask={mask_flag}\n{'='*70}")
        try:
            vm = VascuMap(
                image_source_path=str(file_path), image_index=idx,
                device_width_um=device_width_um, mask_central_region=mask_flag,
                brightfield_channel=brightfield_channel, model_p2p=model_p2p,
                model_unet=model_unet, failure_output_dir=str(failure_diag_dir),
            )
            vm.image_name = name
            output_dir = Path(output_base) / name
            print(f"  Output → {output_dir}")
            vm.pipeline(output_dir=output_dir, save_all_interim=save_all_interim)
            results.append((name, "OK", ""))
        except Exception as exc:
            print(f"  [SKIP] {exc}")
            results.append((file_path.name + lif_tag, "FAILED", str(exc)))
    succeeded = sum(1 for _, s, _ in results if s == "OK")
    print(f"\n{'='*70}\nBatch complete: {succeeded}/{len(results)} succeeded")
    for name, status, msg in results:
        if status == "FAILED":
            print(f"  FAILED: {name}: {msg}")
    return results

In [3]:
shared_model_p2p = Pix2Pix(
    model_path=r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\epoch=117-val_g_psnr=20.47-val_g_ssim=0.62.ckpt"
)
shared_model_unet = load_segmentation_model(
    r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\best_full.pth"
)


In [4]:
# ── Configuration (edit here) ──────────────────────────────────────────────────
source_dir  = r"Z:\Avalon\Devices_mainpage\Thunder\20260416_sexdiff_vascumap"
output_base = r"Z:\Bel\Individual_Vascumap_Outputs\Avalon_Vascumap"

use_gui               = False    # True = napari curation, False = fully automatic
device_width_um       = 35.0
brightfield_channel   = 0
force_mask            = None     # "dark" / "light" / False / None (auto)
require_merged        = True
save_all_interim      = False

In [5]:
# ── Discover jobs ────────────────────────────────────────────────────────────
source, image_files, all_jobs = discover_jobs(source_dir, force_mask, require_merged)
print(f"Source: {source}")
print(f"Files:  {len(image_files)}  |  Total jobs: {len(all_jobs)}")

# Collect folders that are already done (perfect + output_base)
skip_dirs = []
perfect_dir = Path(r"Z:\Bel\Individual_Vascumap_Outputs\Marina_Vascumap\Outputs\CATEGORISED\Perfect")
if perfect_dir.is_dir():
    skip_dirs.append(perfect_dir)
out_dir = Path(output_base)
if out_dir.is_dir():
    skip_dirs.append(out_dir)

skip_names = set()
for d in skip_dirs:
    skip_names |= {f.name for f in d.iterdir() if f.is_dir()}
print(f"Found {len(skip_names)} already-processed folders")

# Filter out already-done jobs
jobs = filter_jobs(all_jobs, skip_names)

for i, (file_path, idx, mask, image_name) in enumerate(jobs, 1):
    lif_tag = (f" (LIF image {idx}: '{image_name}')"
             if file_path.suffix.lower() == ".lif" else "")
    print(f"  {i}. {file_path.name}{lif_tag}  mask={mask}")

Source: Z:\Avalon\Devices_mainpage\Thunder\20260416_sexdiff_vascumap
Files:  6  |  Total jobs: 28
Found 152 already-processed folders
  1. 20260416_Bloom.lif (LIF image 5: 'Bloom_F_D1_Merged')  mask=False
  2. 20260416_Bloom.lif (LIF image 6: 'R 2_Merged')  mask=False
  3. 20260416_Bloom.lif (LIF image 7: 'R 3_Merged')  mask=False
  4. 20260416_Bloom.lif (LIF image 8: 'R 4_Merged')  mask=False
  5. 20260416_Bloom.lif (LIF image 9: 'R 5_Merged')  mask=False
  6. 20260416_Daisy.lif (LIF image 4: 'R 1_Merged')  mask=False
  7. 20260416_Daisy.lif (LIF image 5: 'R 2_Merged')  mask=False
  8. 20260416_Daisy.lif (LIF image 6: 'R 3_Merged')  mask=False
  9. 20260416_Daisy.lif (LIF image 7: 'R 4_Merged')  mask=False
  10. 20260416_Iris.lif (LIF image 5: 'R 1_Merged')  mask=False
  11. 20260416_Iris.lif (LIF image 6: 'R 2_Merged')  mask=False
  12. 20260416_Iris.lif (LIF image 7: 'R 3_Merged')  mask=False
  13. 20260416_Iris.lif (LIF image 8: 'R 4_Merged')  mask=False
  14. 20260416_Iris.lif (LI

In [6]:
# ── GUI curation + batch pipeline ─────────────────────────────────────────────
# Run this cell, curate images in napari, then click "Done — Start Batch" in
# the right-hand panel. Finalisation and the batch pipeline run automatically
# after you click the button — nothing else needs to be run manually.
if use_gui:
    def _on_curation_done(curated_jobs):
        run_batch_from_curation(curated_jobs, output_base, save_all_interim,
                                shared_model_p2p, shared_model_unet)

    app = CurationApp(jobs, device_width_um=device_width_um,
                      on_done=_on_curation_done)
    app.open()
    # Napari viewer is now open.
    # Navigate: n = next, b = back, a = accept, s = skip
    # Edit the device ROI (red polygon) and organoid mask (labels layer).
    # When finished, click "Done — Start Batch" in the panel on the right.
else:
    run_batch(jobs, output_base, device_width_um, brightfield_channel,
              save_all_interim, shared_model_p2p, shared_model_unet)



[1/28] 20260416_Bloom.lif (LIF #5: Bloom_F_D1_Merged)  mask=False
  Output → Z:\Bel\Individual_Vascumap_Outputs\Avalon_Vascumap\20260416_Bloom_Bloom_F_D1_Merged_img5
  Preprocess: 48.9 s


Processing chunks: 100%|██████████| 3/3 [00:14<00:00,  4.89s/it]


  Model inference: 551.8 s
  Postprocess: 0.0 s
  Trim: 0.1 s


Processing chunks: 100%|██████████| 1/1 [00:01<00:00,  1.31s/it]


 chip_volume_um3  convex_hull_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_orientation_deg  p90_minus_p10_sprout_and_branch_orientation_deg  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_sprout_and_branch_length_um  p90_minus_p10_sprout_and_branch_length_um  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  average_vessel_volume_um3  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3  total_number_of_sprouts  total_number_of_branches  total_number_of_junctions  total_number_of_edges  tot

...

 chip_volume_um3  convex_hull_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_orientation_deg  p90_minus_p10_sprout_and_branch_orientation_deg  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_sprout_and_branch_length_um  p90_minus_p10_sprout_and_branch_length_um  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  average_vessel_volume_um3  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3  total_number_of_sprouts  total_number_of_branches  total_number_of_junctions  total_number_of_edges  tot

Processing chunks: 100%|██████████| 3/3 [00:13<00:00,  4.49s/it]


  Model inference: 517.8 s
  Postprocess: 0.0 s
  Trim: 0.1 s


Processing chunks: 100%|██████████| 1/1 [00:01<00:00,  1.50s/it]


 chip_volume_um3  convex_hull_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_orientation_deg  p90_minus_p10_sprout_and_branch_orientation_deg  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_sprout_and_branch_length_um  p90_minus_p10_sprout_and_branch_length_um  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  average_vessel_volume_um3  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3  total_number_of_sprouts  total_number_of_branches  total_number_of_junctions  total_number_of_edges  tot